# 00 — First principles of AI-powered recruitment

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~30 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/08_recruitment_matching/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and NLP familiarity (embeddings, similarity)
- Understanding of text processing and classification

**What you will learn**:
- Why candidate matching is a *semantic similarity with constraints* problem
- Why keyword matching systematically rejects qualified candidates and amplifies bias
- How embeddings capture skill relationships that keywords miss entirely
- That structured rubrics dramatically reduce evaluation inconsistency
- Why every hiring recommendation needs an auditable explanation
- The $200 B+ recruiting market opportunity and the diversity imperative

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.2.0" "numpy>=1.24.0" "scikit-learn>=1.2.0" "matplotlib>=3.7.0"

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Set
from collections import defaultdict
import re, textwrap, json

np.random.seed(42)
print("Setup complete ✓")

## Section 1 — What is candidate matching?

Candidate matching is **semantic similarity with constraints**: given a set of
job requirements *(skills, experience, culture)* and a candidate profile
*(skills, experience, background)*, determine how well they align — and *explain
why*.

Unlike simple document similarity, matching is **asymmetric** and
**multi-dimensional**: a job may *require* Python but only *prefer* Rust, and a
candidate with 10 years of C++ is not the same as one with 10 years of
JavaScript, even if both are "experienced engineers."

Let's define the core abstractions and evaluate five job–candidate pairs at
varying quality levels.

In [ ]:
@dataclass
class Skill:
    """A single skill with proficiency level."""
    name: str
    years: float = 0.0
    proficiency: str = "intermediate"  # beginner, intermediate, advanced, expert

@dataclass
class JobRequirement:
    """Structured job requirement with priority."""
    skill: str
    min_years: float = 0.0
    priority: str = "required"  # required | preferred | nice-to-have
    category: str = "technical"  # technical | experience | education | culture

@dataclass
class CandidateProfile:
    """Structured candidate profile."""
    name: str
    skills: List[Skill] = field(default_factory=list)
    years_experience: float = 0.0
    education: str = ""
    background: str = ""

def match_score(job: List[JobRequirement], candidate: CandidateProfile) -> Dict[str, float]:
    """Compute naive match: fraction of required skills found in candidate."""
    candidate_skills: Set[str] = {s.name.lower() for s in candidate.skills}
    required = [r for r in job if r.priority == "required"]
    preferred = [r for r in job if r.priority == "preferred"]

    req_hits = sum(1 for r in required if r.skill.lower() in candidate_skills)
    pref_hits = sum(1 for r in preferred if r.skill.lower() in candidate_skills)

    req_score = req_hits / len(required) if required else 1.0
    pref_score = pref_hits / len(preferred) if preferred else 1.0
    overall = 0.7 * req_score + 0.3 * pref_score
    return {"required": req_score, "preferred": pref_score, "overall": overall}

# ── Five job–candidate pairs at varying quality ──
job_reqs = [
    JobRequirement("Python", 3, "required"),
    JobRequirement("Machine Learning", 2, "required"),
    JobRequirement("SQL", 1, "required"),
    JobRequirement("Kubernetes", 1, "preferred"),
    JobRequirement("Communication", 0, "preferred", "culture"),
]

candidates = [
    CandidateProfile("Alice", [Skill("Python", 5), Skill("Machine Learning", 3),
                                Skill("SQL", 4), Skill("Kubernetes", 2),
                                Skill("Communication", 5)], 6, "MS CS", "Tech lead"),
    CandidateProfile("Bob", [Skill("Python", 4), Skill("Machine Learning", 2),
                              Skill("SQL", 2)], 4, "BS CS", "Backend dev"),
    CandidateProfile("Carol", [Skill("R", 5), Skill("Statistics", 4),
                                Skill("MATLAB", 3)], 8, "PhD Physics", "Researcher"),
    CandidateProfile("David", [Skill("Java", 6), Skill("Spring", 4),
                                Skill("Oracle", 3)], 7, "BS SE", "Enterprise dev"),
    CandidateProfile("Eve", [Skill("Python", 2), Skill("TensorFlow", 1)],
                     1, "Bootcamp", "Career changer"),
]

print("=" * 60)
print("  NAIVE KEYWORD MATCH — 5 CANDIDATES")
print("=" * 60)
for c in candidates:
    scores = match_score(job_reqs, c)
    bar = "█" * int(scores["overall"] * 20) + "░" * (20 - int(scores["overall"] * 20))
    print(f"  {c.name:<8} {bar}  {scores['overall']:.0%}  "
          f"(req {scores['required']:.0%}, pref {scores['preferred']:.0%})")
print()
print("⚠ Carol (PhD Physics, 8 yrs) scores 0 % — keyword matching")
print("  doesn't understand that R ≈ Python or Statistics ≈ ML.")

## Section 2 — Why keyword matching fails and causes bias

Applicant Tracking Systems (ATS) reject **75 % of resumes** before a human sees
them, relying on keyword overlap. This creates two cascading failures:

1. **Qualified candidates rejected** — A Physics PhD who does statistical
   modelling in R is a strong data-science candidate, but "Python" ≠ "R" and
   "Machine Learning" ≠ "Statistical Modelling."
2. **Demographic bias amplified** — Candidates from different educational
   traditions, countries, or career paths use *different terminology* for
   equivalent skills. Keyword matching penalises linguistic diversity.

In [ ]:
def keyword_match(job_text: str, resume_text: str) -> float:
    """Jaccard similarity on lowercased word sets."""
    job_words: Set[str] = set(job_text.lower().split())
    res_words: Set[str] = set(resume_text.lower().split())
    intersection = job_words & res_words
    union = job_words | res_words
    return len(intersection) / len(union) if union else 0.0

# ── Job description ──
jd = ("Senior Data Scientist with Python, Machine Learning, "
      "SQL, deep learning, NLP, and Kubernetes experience")

# ── Two equally qualified candidates, different terminology ──
resume_match = ("Data Scientist with Python, Machine Learning, "
                "SQL, deep learning, NLP, and Kubernetes")
resume_physics = ("Computational physicist with statistical modelling, "
                  "numerical computing, R, MATLAB, data analysis, "
                  "pattern recognition, and cluster computing")
resume_intl = ("Machine learning ingénieur with Python, apprentissage "
               "automatique, bases de données SQL, deep learning, "
               "traitement du langage naturel, orchestration conteneurs")

results: List[Tuple[str, str, float]] = [
    ("Exact-terminology", resume_match, keyword_match(jd, resume_match)),
    ("Physics PhD", resume_physics, keyword_match(jd, resume_physics)),
    ("International (FR)", resume_intl, keyword_match(jd, resume_intl)),
]

print("=" * 60)
print("  KEYWORD MATCHING — TERM MISMATCH DEMO")
print("=" * 60)
for label, _, score in results:
    bar = "█" * int(score * 40) + "░" * (40 - int(score * 40))
    print(f"  {label:<20} {bar}  {score:.1%}")

# ── Recall gap ──
qualified = 3   # all three are qualified by an expert
keyword_pass = sum(1 for _, _, s in results if s > 0.20)
recall = keyword_pass / qualified
print(f"
  Keyword recall on qualified candidates: {recall:.0%}")
print(f"  ⚠ {qualified - keyword_pass} of {qualified} qualified candidates rejected")
print("  → Keyword matching creates a recall gap that correlates with background.")

## Section 3 — Semantic skill understanding

Embeddings place related concepts close together in vector space. "Container
orchestration," "Kubernetes," "Docker Swarm," and "ECS" all cluster together —
as do "Machine Learning," "Statistical Modelling," and "Pattern Recognition."

This lets us build a **skill similarity matrix** that captures relationships
keyword matching completely misses.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Encoder loaded ✓")

# ── Skills to compare ──
skills: List[str] = [
    "Python programming",
    "R statistical computing",
    "Machine Learning",
    "Statistical Modelling",
    "Kubernetes",
    "Docker Swarm",
    "Container orchestration",
    "Amazon ECS",
    "Deep learning",
    "Pattern recognition",
]

embeddings = encoder.encode(skills)
sim_matrix: np.ndarray = cosine_similarity(embeddings)

# ── Print similarity matrix ──
print("\n" + "=" * 80)
print("  SEMANTIC SKILL SIMILARITY MATRIX")
print("=" * 80)
header = "                       " + "  ".join(f"{s[:6]:>6}" for s in skills)
print(header)
for i, skill in enumerate(skills):
    row = f"  {skill:<22}" + "  ".join(
        f"{sim_matrix[i][j]:6.2f}" for j in range(len(skills))
    )
    print(row)

# ── Compare semantic vs keyword for specific pairs ──
pairs: List[Tuple[str, str]] = [
    ("Python programming", "R statistical computing"),
    ("Machine Learning", "Statistical Modelling"),
    ("Kubernetes", "Container orchestration"),
    ("Deep learning", "Pattern recognition"),
]

print("\n" + "-" * 60)
print("  SEMANTIC vs KEYWORD — selected pairs")
print("-" * 60)
for a, b in pairs:
    idx_a, idx_b = skills.index(a), skills.index(b)
    sem = sim_matrix[idx_a][idx_b]
    kw = keyword_match(a, b)
    print(f"  {a:<25} ↔ {b:<25}")
    print(f"    Semantic: {sem:.3f}   Keyword: {kw:.3f}   "
          f"Gap: {sem - kw:+.3f}")
print("\n✓ Semantic similarity captures skill relationships that keywords miss.")

## Section 4 — Fair evaluation rubrics

Unstructured interviews and ad-hoc resume screening produce wildly inconsistent
results. Research shows **inter-rater reliability** for unstructured evaluations
hovers around κ = 0.2 (poor), while structured rubrics achieve κ > 0.6 (substantial).

A rubric defines:
- **Criteria** — what to evaluate (skills, experience, education, culture signals)
- **Scale** — how to score (1–5 with anchored descriptions)
- **Weights** — relative importance of each criterion

In [ ]:
@dataclass
class RubricCriterion:
    """Single evaluation criterion with scale anchors."""
    name: str
    weight: float
    anchors: Dict[int, str]  # score → description

@dataclass
class CandidateScore:
    """Score for one candidate on one criterion."""
    criterion: str
    score: int
    evidence: str
    reasoning: str

def build_hiring_rubric() -> List[RubricCriterion]:
    """Build a structured hiring rubric with anchored scales."""
    return [
        RubricCriterion("Technical Skills", 0.35, {
            1: "No relevant technical skills demonstrated",
            2: "Basic awareness of required technologies",
            3: "Working knowledge, can contribute with mentoring",
            4: "Strong proficiency, independent contributor",
            5: "Expert level, can mentor others and lead design",
        }),
        RubricCriterion("Experience Depth", 0.25, {
            1: "No relevant professional experience",
            2: "< 1 year or tangentially related",
            3: "2-3 years directly relevant",
            4: "4-6 years with increasing responsibility",
            5: "7+ years with leadership and architecture",
        }),
        RubricCriterion("Education & Learning", 0.15, {
            1: "No relevant education or self-study",
            2: "Some coursework or certifications",
            3: "Degree in related field or extensive self-study",
            4: "Advanced degree or deep specialisation",
            5: "PhD or equivalent research contribution",
        }),
        RubricCriterion("Culture & Communication", 0.15, {
            1: "No evidence of collaboration or communication",
            2: "Basic communication demonstrated",
            3: "Clear communicator, team contributor",
            4: "Strong collaborator, cross-functional experience",
            5: "Exceptional communicator, community builder",
        }),
        RubricCriterion("Growth Potential", 0.10, {
            1: "No evidence of learning or adaptability",
            2: "Some interest in growth",
            3: "Active learner, adapts to new challenges",
            4: "Rapid learner, seeks stretch assignments",
            5: "Visionary, drives innovation and mentors others",
        }),
    ]

rubric = build_hiring_rubric()

# ── Simulate structured vs unstructured evaluation ──
# Unstructured: 4 raters give overall 1-5, high variance
unstructured_ratings = np.array([
    [4, 2, 3, 5],  # candidate A — 4 raters
    [3, 4, 2, 3],  # candidate B
    [5, 3, 4, 2],  # candidate C
])

# Structured: same 4 raters using rubric, lower variance
structured_ratings = np.array([
    [4, 3, 4, 4],  # candidate A
    [3, 3, 3, 3],  # candidate B
    [4, 4, 3, 4],  # candidate C
])

print("=" * 60)
print("  RUBRIC CONSISTENCY — STRUCTURED vs UNSTRUCTURED")
print("=" * 60)
for i, label in enumerate(["Candidate A", "Candidate B", "Candidate C"]):
    u_std = np.std(unstructured_ratings[i])
    s_std = np.std(structured_ratings[i])
    reduction = (u_std - s_std) / u_std * 100 if u_std > 0 else 0
    print(f"  {label}:")
    print(f"    Unstructured σ = {u_std:.2f}  Structured σ = {s_std:.2f}  "
          f"Variance reduction: {reduction:.0f} %")

# ── Display rubric ──
print("\n" + "-" * 60)
print("  HIRING RUBRIC")
print("-" * 60)
for crit in rubric:
    print(f"\n  {crit.name} (weight: {crit.weight:.0%})")
    for score, desc in sorted(crit.anchors.items()):
        print(f"    {score}: {desc}")
print("\n✓ Structured rubrics reduce rater variance by 50-70 %.")

## Section 5 — Explainability in hiring

In hiring, every recommendation **must** come with a rationale. Candidates
deserve to know *why* they were ranked. Hiring managers need evidence to make
informed decisions. Regulators (EU AI Act, NYC Local Law 144) increasingly
*require* explanations for automated hiring tools.

A good explanation follows: **criterion → evidence → score → reasoning**.

In [ ]:
@dataclass
class MatchExplanation:
    """Structured explanation for a hiring recommendation."""
    candidate_name: str
    job_title: str
    overall_score: float
    criterion_scores: List[CandidateScore]
    recommendation: str  # strong-yes | yes | maybe | no | strong-no
    summary: str

def format_explanation(exp: MatchExplanation) -> str:
    """Format a match explanation as a readable scorecard."""
    lines: List[str] = [
        f"╔{'═' * 58}╗",
        f"║  CANDIDATE SCORECARD{' ' * 38}║",
        f"╠{'═' * 58}╣",
        f"║  Candidate : {exp.candidate_name:<43}║",
        f"║  Position  : {exp.job_title:<43}║",
        f"║  Overall   : {exp.overall_score:.1f}/5.0  →  {exp.recommendation:<28}║",
        f"╠{'═' * 58}╣",
    ]
    for cs in exp.criterion_scores:
        lines.append(f"║  {cs.criterion:<20} : {cs.score}/5{' ' * 30}║")
        lines.append(f"║    Evidence  : {cs.evidence[:40]:<43}║")
        lines.append(f"║    Reasoning : {cs.reasoning[:40]:<43}║")
    lines.append(f"╠{'═' * 58}╣")
    lines.append(f"║  Summary: {exp.summary[:46]:<47}║")
    lines.append(f"╚{'═' * 58}╝")
    return "\n".join(lines)

# ── Good explanation vs bad explanation ──
good = MatchExplanation(
    candidate_name="Alice Chen",
    job_title="Senior Data Scientist",
    overall_score=4.2,
    criterion_scores=[
        CandidateScore("Technical Skills", 5,
                       "5 yrs Python, 3 yrs ML, TF + PyTorch",
                       "Expert in all required technologies"),
        CandidateScore("Experience Depth", 4,
                       "6 yrs including team lead at FinTech",
                       "Strong IC-to-lead trajectory"),
        CandidateScore("Education", 4,
                       "MS Computer Science, Stanford",
                       "Advanced degree in relevant field"),
        CandidateScore("Culture", 4,
                       "Led 3 cross-team projects, mentored 5",
                       "Strong collaboration evidence"),
        CandidateScore("Growth", 3,
                       "Published 2 papers, speaks at meetups",
                       "Active in community, moderate innovation"),
    ],
    recommendation="strong-yes",
    summary="Exceptional technical fit with leadership potential."
)

bad_summary = "Looks like a good candidate. Seems smart."

print("✓ GOOD EXPLANATION (structured, evidence-based):")
print(format_explanation(good))
print()
print("✗ BAD EXPLANATION (vague, no evidence):")
print(f'  "{bad_summary}"')
print("  → No criteria, no evidence, no reasoning, potential bias.")

## Section 6 — Market analysis

The recruiting industry is massive and ripe for AI transformation:

| Metric | Value | Source |
|--------|-------|--------|
| Global staffing market | $200 B+ | SIA, 2024 |
| Average cost per hire | $4,700 | SHRM, 2023 |
| Time to fill (avg) | 44 days | SHRM, 2023 |
| Screening time per hire | 23 hours | Ideal, 2023 |
| ATS rejection rate | 75 % of applicants | Harvard Business School, 2021 |
| Eightfold AI valuation | $2.1 B | TechCrunch, 2023 |

The business case for AI recruitment matching rests on three pillars:
**speed** (reduce 23 hrs → minutes), **quality** (semantic understanding vs keywords),
and **fairness** (structured evaluation reduces bias).

In [ ]:
# ── Market sizing and cost model ──
manual_hours_per_hire: float = 23.0
hourly_rate: float = 45.0  # recruiter loaded cost
manual_cost: float = manual_hours_per_hire * hourly_rate

ai_minutes_per_hire: float = 15.0
ai_cost_per_hire: float = 2.50  # API + compute
hires_per_year: int = 500

manual_annual = hires_per_year * manual_cost
ai_annual = hires_per_year * ai_cost_per_hire
savings = manual_annual - ai_annual
time_saved = hires_per_year * (manual_hours_per_hire - ai_minutes_per_hire / 60)

print("=" * 60)
print("  RECRUITMENT AI — ROI MODEL")
print("=" * 60)
print(f"  Manual cost/hire   : ${manual_cost:,.0f}  ({manual_hours_per_hire}h × ${hourly_rate}/h)")
print(f"  AI cost/hire       : ${ai_cost_per_hire:,.2f}")
print(f"  Savings/hire       : ${manual_cost - ai_cost_per_hire:,.0f}")
print(f"  Annual hires       : {hires_per_year:,}")
print(f"  Manual annual cost : ${manual_annual:,.0f}")
print(f"  AI annual cost     : ${ai_annual:,.0f}")
print(f"  Annual savings     : ${savings:,.0f}  ({savings / manual_annual:.0%})")
print(f"  Hours reclaimed    : {time_saved:,.0f} hours/year")

# ── Quality improvement ──
print("\n" + "-" * 60)
print("  QUALITY IMPROVEMENT")
print("-" * 60)
metrics: List[Tuple[str, float, float]] = [
    ("Qualified-candidate recall", 0.25, 0.85),
    ("Screening consistency (κ)", 0.20, 0.65),
    ("Time to shortlist", 23.0, 0.25),  # hours
    ("Adverse impact ratio", 0.60, 0.88),
]
for name, before, after in metrics:
    arrow = "↑" if after > before else "↓"
    print(f"  {name:<30} {before:>8.2f} → {after:>8.2f}  {arrow}")

print("\n✓ AI recruitment matching: faster, fairer, and more accurate.")

In [ ]:
# ── Visualise keyword vs semantic matching ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: keyword scores for 3 candidate types
labels = [r[0] for r in results]
kw_scores = [r[2] for r in results]
colors = ["#388e3c", "#d32f2f", "#f57c00"]
axes[0].barh(labels, kw_scores, color=colors, height=0.5)
axes[0].set_xlim(0, 1.0)
axes[0].set_xlabel("Keyword overlap (Jaccard)")
axes[0].set_title("Keyword matching — term mismatch problem")
axes[0].axvline(x=0.20, color="gray", linestyle="--", alpha=0.5, label="Pass threshold")
axes[0].legend()

# Heatmap-style bar chart: rubric variance reduction
cand_labels = ["Candidate A", "Candidate B", "Candidate C"]
u_stds = [np.std(unstructured_ratings[i]) for i in range(3)]
s_stds = [np.std(structured_ratings[i]) for i in range(3)]
x = np.arange(len(cand_labels))
axes[1].bar(x - 0.15, u_stds, 0.3, label="Unstructured", color="#d32f2f", alpha=0.8)
axes[1].bar(x + 0.15, s_stds, 0.3, label="Structured", color="#388e3c", alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(cand_labels)
axes[1].set_ylabel("Rating std dev (σ)")
axes[1].set_title("Rubric consistency — structured vs unstructured")
axes[1].legend()

plt.tight_layout()
plt.show()
print("✓ Visualisations rendered")

## Exercises

1. **Expand the skill graph** — Add 10 more skills to the similarity matrix in
   Section 3 (e.g., "PyTorch," "scikit-learn," "data visualisation"). Identify
   clusters of related skills and calculate the average intra-cluster similarity.

2. **Build a weighted rubric scorer** — Using the rubric from Section 4, write a
   function `score_candidate(rubric, scores) -> float` that computes a weighted
   overall score. Test it with at least 3 candidates and compare rankings.

3. **Bias simulation** — Create 20 synthetic resumes: 10 using US tech
   terminology and 10 using equivalent international terms. Run keyword matching
   and semantic matching on both groups. Calculate the recall gap and adverse
   impact ratio for each method.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Candidate matching is semantic similarity with constraints, not keyword overlap |
| 2 | ATS keyword matching rejects 75 % of applicants and disproportionately hurts diverse candidates |
| 3 | Embeddings capture skill relationships (Python ↔ R, Kubernetes ↔ Docker Swarm) that keywords miss |
| 4 | Structured rubrics reduce inter-rater variance by 50–70 % compared to unstructured evaluation |
| 5 | Every hiring recommendation needs criterion → evidence → score → reasoning |
| 6 | The $200 B+ recruiting market has a clear business case: faster, fairer, cheaper |
| 7 | Regulatory pressure (EU AI Act, NYC LL144) makes explainability a legal requirement |

## What's Next

In **01 — Architecture**, we design the end-to-end recruitment matching pipeline:
JD parsing, resume understanding, semantic matching, rubric evaluation, bias
detection, and scorecard generation.

---

## References

1. SHRM, *Average Cost-Per-Hire for Companies Is $4,700*, 2023 — <https://www.shrm.org/>
2. Fuller, J. et al., *Hidden Workers: Untapped Talent*, Harvard Business School, 2021 — <https://www.hbs.edu/>
3. Eightfold AI — <https://eightfold.ai/>
4. EU AI Act, *Regulation (EU) 2024/1689 — Artificial Intelligence Act*, 2024 — <https://artificialintelligenceact.eu/>